# NB06 — Luki-style SAM1 recovery | Baego
**Goal:** visually inspect the original successful Block 53 SAM1 ViT-H predictions and NMS alternative, already saved for Baego ROI 1 and ROI 4. No GPU, inference, training, or destructive writes.
The Luki site remains part of the project but is **not displayed until its own verified saved predictions are identified**. Reported F1 values come from archived CSV, not fresh evaluation.

In [ ]:
# 01 — Setup (read-only paths)
from pathlib import Path
from google.colab import drive
import json, re, warnings
import numpy as np, pandas as pd
import geopandas as gpd
import rasterio
import matplotlib.pyplot as plt
from shapely.geometry import box
from rasterio.windows import from_bounds
from rasterio.windows import transform as window_transform

drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/Colab Notebooks/Musanga')
DATA = ROOT / 'data/Geo_Ref'
ARCHIVE = ROOT / 'KVO_work/KVO_06_UNet_guided_SAM3'
ARCHIVE_V2 = ROOT / 'KVO_work/KVO_06_v2_Baego_Luki'
INPUTS = {'Baego_RGB': DATA/'Baego_georef.tif', 'Baego_Kristof': DATA/'Annotations/Crown_Kristof.shp', 'Baego_Nadine': DATA/'Annotations/Crown_Nadine.shp', 'Baego_ROI': DATA/'Annotations/ROI_Crowns_Kvo.shp', 'Luki_RGB': DATA/'Luki.tif', 'Luki_reference': DATA/'Annotations/Tree_Annotation_Luki.gpkg'}
for name,p in INPUTS.items(): print(f'{name:18} {"OK" if p.exists() else "MISSING"}  {p}')
print('Historical folders:', [(str(p),p.exists()) for p in (ARCHIVE,ARCHIVE_V2)])

In [ ]:
# 02 — Reference crowns and area rule (metres squared)
MIN_CROWN_M2 = 5.0

def load_layer(path, target_crs, raster_bounds, layer=None):
    g = gpd.read_file(path, layer=layer) if layer else gpd.read_file(path)
    g = g.loc[g.geometry.notna() & ~g.geometry.is_empty].copy()
    if g.empty:
        print(f'WARNING: no geometries in {path.name}')
        return g

    # A missing .prj is common in older annotation shapefiles.
    # Assign raster CRS only when the raw coordinate extents already overlap.
    if g.crs is None:
        xmin, ymin, xmax, ymax = g.total_bounds
        rxmin, rymin, rxmax, rymax = tuple(raster_bounds)
        overlaps = xmax >= rxmin and xmin <= rxmax and ymax >= rymin and ymin <= rymax
        if not overlaps:
            raise ValueError(
                f'{path.name} has no CRS and its coordinates do not overlap the RGB raster. '
                f'Annotation bounds={g.total_bounds}; raster bounds={tuple(raster_bounds)}. '
                'Check the original annotation CRS before proceeding.'
            )
        print(f'WARNING: {path.name} has no CRS; coordinates overlap raster. '
              f'Assuming raster CRS {target_crs}.')
        g = g.set_crs(target_crs, allow_override=True)
    else:
        g = g.to_crs(target_crs)
    return g

def load_crowns(path, target_crs, raster_bounds, layer=None):
    g = load_layer(path, target_crs, raster_bounds, layer)
    if not g.crs.is_projected:
        raise ValueError('A projected metric CRS is required for the 5 m² crown filter')
    if g.crs.axis_info and not all(
        axis.unit_name.lower() in ('metre','meter','metres','meters') for axis in g.crs.axis_info[:2]
    ):
        raise ValueError(f'CRS units are not metres: {g.crs}')
    before = len(g)
    g = g.loc[g.geometry.area >= MIN_CROWN_M2].copy()
    print(f'  {path.name}: {before} polygons → {len(g)} crowns ≥ {MIN_CROWN_M2:g} m²')
    return g

SITES = {}
for site, rgb_key in [('Baego','Baego_RGB'),('Luki','Luki_RGB')]:
    if not INPUTS[rgb_key].exists(): continue
    with rasterio.open(INPUTS[rgb_key]) as ds:
        if ds.crs is None or not ds.crs.is_projected:
            raise ValueError(f'{site}: projected raster CRS required')
        site_info = {'rgb':INPUTS[rgb_key], 'crs':ds.crs,
                     'bounds':ds.bounds, 'res':ds.res}
    crs, bounds = site_info['crs'], site_info['bounds']
    print(f'\n{site}: raster CRS={crs}, resolution={site_info["res"]}')
    if site == 'Baego':
        site_info['Kristof'] = load_crowns(INPUTS['Baego_Kristof'], crs, bounds)
        site_info['Nadine'] = load_crowns(INPUTS['Baego_Nadine'], crs, bounds)
        site_info['roi'] = load_layer(INPUTS['Baego_ROI'], crs, bounds)
    else:
        site_info['reference'] = load_crowns(
            INPUTS['Luki_reference'], crs, bounds,
            layer='photointerpretation_inventory')
    SITES[site] = site_info
    print('Reference counts:', {
        k:len(v) for k,v in site_info.items()
        if isinstance(v,gpd.GeoDataFrame)
    })

In [ ]:
# 03 — Frozen Luki-inspired Block 53 settings (archived provenance)
# No model initialization or inference. These are the settings recovered from prior experiment notes.
LUKI_STYLE = {
    'model': 'SAM1 ViT-H, automatic mask generation',
    'points_per_side': 32,
    'pred_iou_thresh': 0.88,
    'stability_score_thresh': 0.95,
    'core_size_m': 40,
    'context_buffer_each_side_m': 60,
    'context_extent_m': 160,
    'selection': 'mask centroid inside 40 m core',
    'components': 'up to three largest per mask',
    'predicted_area_range_m2': (2, 2500),
    'postprocessing': ['author_style', 'nms_060'],
}
print('Historical Block 53 method (recovered notes):')
for key,value in LUKI_STYLE.items(): print(f'  {key}: {value}')
print('\nSource: archived experiment v53. These are not newly inferred SAM settings.')
print('Reference crown cutoff remains 5 m²; historical predictions are NOT re-filtered.')

In [ ]:
# 04 — Load ONLY the saved Block 53 Baego predictions and comparison metrics
# This exact archive has ROI 1 and ROI 4. No historical outputs are modified.
HISTORY = ARCHIVE / 'experiments_v53_luki_sam_only_replication'
if not HISTORY.is_dir(): raise FileNotFoundError(f'Block 53 archive not found: {HISTORY}')
metric_path = HISTORY / 'luki_style_roi_comparison.csv'
if not metric_path.exists(): raise FileNotFoundError(metric_path)
METRICS = pd.read_csv(metric_path)
print('=== ORIGINAL BLOCK 53: SAVED F1 AT IoU >= 0.5 ===')
display(METRICS.loc[METRICS['iou_threshold'].eq(0.5),
    ['roi','postprocess','reference','predicted','matched','precision','recall','f1']
].sort_values(['roi','postprocess']).reset_index(drop=True).round(3))

PREDICTIONS_BY_ROI = {}
for roi_id in (1,4):
    for method in ('author_style','nms_060'):
        p=HISTORY / f'roi{roi_id}_{method}.gpkg'
        if not p.exists(): raise FileNotFoundError(p)
        g=gpd.read_file(p)
        if g.crs is None:
            raise ValueError(f'Saved predictions missing CRS: {p.name}; do not guess.')
        g=g.to_crs(SITES['Baego']['crs'])
        PREDICTIONS_BY_ROI[(roi_id,method)] = g
        print(f'ROI {roi_id:>1}, {method:12}: {len(g)} saved predictions ({p.name})')

# Check that predictions really overlap the numbered ROI; fail instead of showing unrelated geometry.
baego_rois=SITES['Baego']['roi']
for roi_id in (1,4):
    if roi_id>=len(baego_rois): raise ValueError(f'Missing Baego ROI index {roi_id}')
    roi_geom=baego_rois.geometry.iloc[roi_id]
    for method in ('author_style','nms_060'):
        g=PREDICTIONS_BY_ROI[(roi_id,method)]
        in_roi=g.geometry.intersects(roi_geom).sum()
        print(f'ROI {roi_id} {method}: {in_roi}/{len(g)} saved polygons intersect the current ROI geometry')
        if in_roi==0:
            raise ValueError('ROI order likely changed. Resolve mapping before visualization.')
print('\nReady: Block 05 shows the original author-style and NMS crown boundaries.')

In [ ]:
# 05 — Visual comparison: original Block 53 vs NMS on RGB
# Simply change VIEW_ROI to 1 or 4. No inference, no writes.
VIEW_ROI = 1  # 1 or 4
if VIEW_ROI not in (1,4): raise ValueError('Block 53 has saved outputs only for ROI 1 and ROI 4.')
s=SITES['Baego']
roi_geom=s['roi'].geometry.iloc[VIEW_ROI]
bounds=roi_geom.bounds
with rasterio.open(s['rgb']) as ds:
    from rasterio.windows import Window
    w=from_bounds(*bounds,transform=ds.transform).round_offsets().round_lengths()
    w=w.intersection(Window(0,0,ds.width,ds.height))
    if w.width<=0 or w.height<=0: raise ValueError('ROI lies outside Baego RGB.')
    rgb=ds.read([1,2,3],window=w,
                out_shape=(3,min(int(w.height),1600),min(int(w.width),1600)))
    bb=rasterio.windows.bounds(w,ds.transform)
img=np.moveaxis(rgb,0,-1).astype(float)
lo,hi=np.nanpercentile(img,[2,98])
img=np.clip((img-lo)/(hi-lo+1e-9),0,1)
extent=[bb[0],bb[2],bb[1],bb[3]]
area=box(*bb)
fig,axes=plt.subplots(1,2,figsize=(16,8),constrained_layout=True)
for ax,method,title in zip(axes,('author_style','nms_060'),
                            ('Luki-style SAM1 ViT-H','Luki-style SAM1 + NMS')):
    ax.imshow(img,extent=extent)
    for label,color in (('Kristof','white'),('Nadine','yellow')):
        g=s[label]
        g=g.loc[g.geometry.intersects(area)]
        if len(g): g.boundary.plot(ax=ax,color=color,linewidth=0.8)
    g=PREDICTIONS_BY_ROI[(VIEW_ROI,method)]
    visible=g.loc[g.geometry.intersects(area)]
    if len(visible): visible.boundary.plot(ax=ax,color='red',linewidth=1)
    score=METRICS.loc[(METRICS.roi==VIEW_ROI)&(METRICS.postprocess==method)&
                       (METRICS.iou_threshold==0.5),'f1']
    score_text=f'{float(score.iloc[0]):.3f}' if len(score) else 'n/a'
    ax.set_title(f'{title}\nF1={score_text} | {len(g)} saved crowns')
    ax.set_xlim(bb[0],bb[2]);ax.set_ylim(bb[1],bb[3])
    ax.set_aspect('equal')
fig.suptitle(f'Baego ROI {VIEW_ROI} | white=Kristof, yellow=Nadine, red=SAM')
plt.show()
print('Next: switch VIEW_ROI between 1 and 4. Judge complete individual crowns and seam artifacts.')

In [ ]:
# 06 — Brief assessment summary (original metrics, NOT recalculated)
d=METRICS.loc[METRICS.iou_threshold==0.5]
print('Historical Block 53, IoU >= 0.5')
display(d.pivot(index='roi',columns='postprocess',values='f1').round(3))
print('ROI 1 and 4 only. Do not generalize these scores to Luki or all six Baego ROIs.')
print('Nadine comparisons are a separate sparse-reference benchmark; not pooled with Kristof F1.')
print('Luki is not run in this notebook: the historical Luki-site result set remains to be identified.')
print('No new SAM inference or training; no original outputs modified.')

## 07 — Decision log
- **Recovered method:** SAM1 ViT-H, automatic mask generation, 40 m cores + 60 m context on each side (Block 53).
- **Archived inputs:** `experiments_v53_luki_sam_only_replication/roi{1,4}_{author_style,nms_060}.gpkg`
- **Expected F1 at IoU 0.5:** ROI 1 author-style 0.493 / NMS 0.517; ROI 4 author-style 0.394 / NMS 0.417.
- **Next decision:** visually inspect ROI 1 and ROI 4 (Block 05) before deciding whether any new inference is justified.
- **Pending:** locate and verify original Luki-site saved predictions and exact historical six-ROI baseline.


In [ ]:
# 08 — Nadine reference matching | original SAM1 vs NMS (CPU, read-only)
# Sparse Nadine annotations permit crown-recovery analysis, NOT stand-wide precision/F1.
from scipy.optimize import linear_sum_assignment
from shapely import make_valid
from shapely.geometry import Polygon, MultiPolygon
from shapely.ops import unary_union

MATCH_IOU = 0.5
VALIDATION_ROIS = (1, 4)

def polygon_only(g):
    if g is None or g.is_empty:
        return Polygon()
    if not g.is_valid:
        g = make_valid(g)
    if isinstance(g, (Polygon, MultiPolygon)):
        return g
    parts = [part for part in getattr(g, "geoms", ())
             if isinstance(part, (Polygon, MultiPolygon)) and not part.is_empty]
    return unary_union(parts) if parts else Polygon()

def repaired_copy(gdf):
    out = gdf.copy().reset_index(drop=True)
    out["geometry"] = gpd.GeoSeries(
        [polygon_only(g) for g in out.geometry],
        index=out.index, crs=out.crs
    )
    return out

def crown_iou(a, b):
    if a.is_empty or b.is_empty:
        return 0.0
    try:
        overlap = a.intersection(b).area
    except Exception:
        a, b = polygon_only(a), polygon_only(b)
        overlap = a.intersection(b).area
    union = a.area + b.area - overlap
    return float(overlap / union) if union > 0 else 0.0

def match_one_to_one(ref, pred, min_iou=0.5):
    mat = np.zeros((len(ref), len(pred)), dtype=np.float32)
    if len(ref) and len(pred):
        tree = pred.sindex
        for i, geom in enumerate(ref.geometry):
            if geom.is_empty:
                continue
            for j in tree.query(geom, predicate="intersects"):
                mat[i, j] = crown_iou(geom, pred.geometry.iloc[j])
        # Hungarian assignment, with below-threshold candidates excluded.
        score = np.where(mat >= min_iou, mat, 0.0)
        rows, cols = linear_sum_assignment(score, maximize=True)
        pairs = [(int(i), int(j), float(mat[i, j]))
                 for i, j in zip(rows, cols) if mat[i, j] >= min_iou]
    else:
        pairs = []
    matched_ref = {i for i, j, _ in pairs}
    return {"pairs": pairs, "matched_ref": matched_ref,
            "matched_pred": {j for i, j, _ in pairs},
            "missed_ref": [i for i in range(len(ref)) if i not in matched_ref],
            "matrix": mat}

NADINE_MATCHES = {}
summary = []
for roi_id in VALIDATION_ROIS:
    region = SITES["Baego"]["roi"].geometry.iloc[roi_id]
    # Use the same centroid selection as earlier validation.
    nad = repaired_copy(SITES["Baego"]["Nadine"])
    ref = nad.loc[nad.geometry.centroid.within(region)].reset_index(drop=True)
    for method in ("author_style", "nms_060"):
        pred = repaired_copy(PREDICTIONS_BY_ROI[(roi_id, method)])
        m = match_one_to_one(ref, pred, MATCH_IOU)
        NADINE_MATCHES[(roi_id, method)] = {"ref": ref, "pred": pred, **m}
        ious = [x[2] for x in m["pairs"]]
        summary.append({"ROI": roi_id, "Method": method,
                        "Nadine_crowns": len(ref), "SAM_polygons": len(pred),
                        "Recovered": len(ious), "Missed": len(m["missed_ref"]),
                        "Nadine_recall": len(ious)/len(ref) if len(ref) else np.nan,
                        "Median_IoU_recovered": np.median(ious) if ious else np.nan})
print("Nadine crown recovery at IoU >= 0.5 (not stand-wide precision/F1)")
display(pd.DataFrame(summary).round(3))

In [ ]:
# 09 — Diagnose every missed Nadine crown using existing SAM polygons (CPU)
# Choose 'author_style' or 'nms_060'. No inference or disk writes.
DIAG_METHOD = "nms_060"
ZOOM_PADDING_M = 6.0
MAX_PANELS_PER_ROI = 20  # set lower for shorter output

diagnostic_rows = []
for roi_id in VALIDATION_ROIS:
    d = NADINE_MATCHES[(roi_id, DIAG_METHOD)]
    refs, preds = d["ref"], d["pred"]
    for i in d["missed_ref"]:
        rg = refs.geometry.iloc[i]
        # All 125–157 saved candidates: cheap, and robust even if no overlap.
        scores = np.array([crown_iou(rg, pg) for pg in preds.geometry])
        best_j = int(np.argmax(scores)) if len(scores) else None
        best = preds.geometry.iloc[best_j] if best_j is not None else None
        max_iou = float(scores[best_j]) if best_j is not None else 0.0
        intersection = rg.intersection(best).area if best is not None else 0.0
        ref_fraction = intersection / rg.area if rg.area else 0.0
        size_ratio = best.area / rg.area if best is not None and rg.area else np.nan
        min_distance = float(min((rg.distance(g) for g in preds.geometry), default=np.nan))
        if max_iou == 0:
            diagnosis = "No overlapping SAM polygon"
        elif size_ratio > 1.5 and ref_fraction >= 0.5:
            diagnosis = "Possible merged / oversized mask"
        elif size_ratio < 0.65:
            diagnosis = "Possible fragment / undersized mask"
        elif max_iou >= 0.3:
            diagnosis = "Subthreshold boundary mismatch"
        else:
            diagnosis = "Weak overlap / uncertain"
        diagnostic_rows.append({
            "ROI": roi_id, "Ref_index": i, "Nadine_area_m2": round(rg.area, 1),
            "Best_SAM_index": best_j, "Best_IoU": round(max_iou, 3),
            "Ref_area_covered": round(ref_fraction, 3),
            "SAM_to_ref_area_ratio": round(size_ratio, 2),
            "Nearest_distance_m": round(min_distance, 2),
            "Likely_issue": diagnosis
        })

DIAG = pd.DataFrame(diagnostic_rows)
print(f"=== Missed Nadine crowns, {DIAG_METHOD}, IoU threshold {MATCH_IOU} ===")
if DIAG.empty:
    print("None missed.")
else:
    display(DIAG.sort_values(["ROI", "Best_IoU"], ascending=[True, False]))
    print("\nCandidate issue categories (heuristic, not validated labels):")
    display(pd.crosstab(DIAG["ROI"], DIAG["Likely_issue"]))

# Small RGB crops instead of rendering full orthomosaics repeatedly.
def plot_missed(roi_id):
    d = NADINE_MATCHES[(roi_id, DIAG_METHOD)]
    subset = DIAG.loc[DIAG.ROI == roi_id].head(MAX_PANELS_PER_ROI)
    if subset.empty:
        print(f"ROI {roi_id}: no missed crowns")
        return
    ncols = 3
    nrows = int(np.ceil(len(subset)/ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(15, 4.8*nrows), squeeze=False)
    raster_path = SITES["Baego"]["rgb"]
    with rasterio.open(raster_path) as src:
        for ax, (_, row) in zip(axes.flat, subset.iterrows()):
            idx = int(row.Ref_index)
            geom = d["ref"].geometry.iloc[idx]
            xmin, ymin, xmax, ymax = geom.bounds
            half = max((xmax-xmin)/2, (ymax-ymin)/2, ZOOM_PADDING_M)
            cx, cy = (xmin+xmax)/2, (ymin+ymax)/2
            half += ZOOM_PADDING_M
            bounds = (cx-half, cy-half, cx+half, cy+half)
            window = from_bounds(*bounds, transform=src.transform)
            window = window.intersection(
                rasterio.windows.Window(0, 0, src.width, src.height))
            if window.width <= 0 or window.height <= 0:
                ax.set_title(f"Ref {idx}: outside RGB"); ax.axis("off"); continue
            arr = src.read([1,2,3], window=window,
                           out_shape=(3, 600, 600))
            bb = rasterio.windows.bounds(window, src.transform)
            im = np.moveaxis(arr, 0, -1).astype(float)
            p2,p98 = np.nanpercentile(im, [2,98])
            im = np.clip((im-p2)/(p98-p2+1e-9),0,1)
            ax.imshow(im, extent=[bb[0],bb[2],bb[1],bb[3]])
            area = box(*bb)
            near = d["pred"].loc[d["pred"].geometry.intersects(area)]
            if len(near): near.boundary.plot(ax=ax, color="red", linewidth=1)
            if pd.notna(row.Best_SAM_index):
                best = d["pred"].geometry.iloc[int(row.Best_SAM_index)]
                gpd.GeoSeries([best], crs=d["pred"].crs).boundary.plot(
                    ax=ax, color="cyan", linewidth=2)
            gpd.GeoSeries([geom], crs=d["ref"].crs).boundary.plot(
                ax=ax, color="white", linewidth=2)
            ax.set_xlim(bb[0],bb[2]); ax.set_ylim(bb[1],bb[3])
            ax.set_title(f"ROI {roi_id} | Nadine #{idx} | best IoU {row.Best_IoU:.2f}\n{row.Likely_issue}", fontsize=9)
            ax.set_xticks([]);ax.set_yticks([]); ax.set_aspect("equal")
        for ax in axes.flat[len(subset):]: ax.axis("off")
    fig.suptitle("White: missed Nadine | cyan: best-IoU SAM | red: other SAM", fontsize=12)
    plt.tight_layout()
    plt.show()

for roi_id in VALIDATION_ROIS:
    plot_missed(roi_id)

print("All diagnostics use archived polygons. Green/recovered crowns are excluded here.")

In [ ]:
# 10 — Did postprocessing discard individual crowns? (CPU, read-only)
# Uses saved raw, author_style, and nms_060 polygons for Baego ROI 1 & 4.
# Requires Blocks 01, 02, 04, 08. Block 09 is optional.
# These are *candidate-mask comparisons*, not new SAM predictions.

from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import rasterio
from rasterio.windows import from_bounds, Window
from shapely.geometry import box

IOU_MATCH = 0.5
INSPECT_PER_ROI = 3       # number of missed Nadine crowns to plot
CHECK_ROIS = (1, 4)

# Load and repair archive COPIES in memory; preserve source files.
SETS = {}
for roi_id in CHECK_ROIS:
    for stage, suffix in (
        ("raw", "luki_style_raw"),
        ("author", "author_style"),
        ("nms", "nms_060")
    ):
        p = HISTORY / f"roi{roi_id}_{suffix}.gpkg"
        if not p.exists():
            print("Missing saved stage:", p)
            continue
        g = gpd.read_file(p)
        if g.crs is None:
            raise ValueError(f"Unknown CRS in {p}; will not assume one")
        g = repaired_copy(g.to_crs(SITES["Baego"]["crs"]))
        g = g.loc[~g.geometry.is_empty].reset_index(drop=True)
        SETS[(roi_id, stage)] = g
        print(f"ROI {roi_id} {stage:>6}: {len(g)} saved polygons")

def overlap_candidates(reference_geometry, candidates):
    """Return candidate index, IoU, reference-cover fraction, and area ratio."""
    if len(candidates) == 0 or reference_geometry.is_empty:
        return []
    ids = candidates.sindex.query(reference_geometry, predicate="intersects")
    result = []
    for j in ids:
        polygon = candidates.geometry.iloc[int(j)]
        if polygon.is_empty:
            continue
        inter = reference_geometry.intersection(polygon).area
        if inter <= 0:
            continue
        union = reference_geometry.area + polygon.area - inter
        result.append({
            "index": int(j),
            "iou": inter / union if union > 0 else 0,
            "ref_covered": inter / reference_geometry.area if reference_geometry.area > 0 else 0,
            "area_ratio": polygon.area / reference_geometry.area if reference_geometry.area > 0 else np.nan
        })
    return sorted(result, key=lambda x: x["iou"], reverse=True)

def near_equivalent(crown, polygons, cutoff=0.8):
    """Check whether essentially the same polygon survives in the next stage."""
    best_iou = 0.0
    if len(polygons) and not crown.is_empty:
        for j in polygons.sindex.query(crown, predicate="intersects"):
            best_iou = max(best_iou, crown_iou(crown, polygons.geometry.iloc[int(j)]))
    return best_iou >= cutoff, best_iou

STAGE_ROWS = []
DETAILS = {}
for roi_id in CHECK_ROIS:
    base = NADINE_MATCHES[(roi_id, "nms_060")]
    refs = base["ref"]
    missed_ids = base["missed_ref"]
    for ref_id in missed_ids:
        rg = refs.geometry.iloc[ref_id]
        stage_hits = {
            stage: overlap_candidates(rg, SETS[(roi_id, stage)])
            if (roi_id, stage) in SETS else []
            for stage in ("raw", "author", "nms")
        }
        DETAILS[(roi_id, ref_id)] = stage_hits
        r = {"ROI": roi_id, "Nadine_index": ref_id, "Nadine_area_m2": round(rg.area, 1)}
        for stage in ("raw", "author", "nms"):
            hits = stage_hits[stage]
            r[f"{stage}_candidates"] = len(hits)
            r[f"{stage}_maxIoU"] = round(hits[0]["iou"], 3) if hits else 0.0
            r[f"{stage}_anyIoU05"] = any(h["iou"] >= IOU_MATCH for h in hits)
            r[f"{stage}_best_cover"] = round(hits[0]["ref_covered"], 3) if hits else 0.0

        # Flag if a *valid candidate* appeared in a prior stage, but there is
        # no >=0.5 candidate in a later stage. This does NOT prove which filter caused removal.
        if r["raw_anyIoU05"] and not r["author_anyIoU05"]:
            issue = "Raw candidate lost before author-style"
        elif r["author_anyIoU05"] and not r["nms_anyIoU05"]:
            issue = "Candidate lost between author-style and NMS"
        elif r["nms_anyIoU05"]:
            issue = "Candidate exists in NMS; one-to-one assignment conflict"
        elif r["raw_candidates"] == 0:
            issue = "No overlapping raw mask"
        else:
            issue = "No stage has an IoU>=0.5 candidate"
        r["diagnosis"] = issue
        STAGE_ROWS.append(r)

STAGE_DIAG = pd.DataFrame(STAGE_ROWS)
print("\n=== TRACKING MISSED NADINE CROWNS THROUGH THREE SAVED STAGES ===")
if len(STAGE_DIAG):
    display(STAGE_DIAG.sort_values(["ROI", "Nadine_index"]).reset_index(drop=True))
    print("\nCounts by ROI and diagnosis:")
    display(pd.crosstab(STAGE_DIAG["ROI"], STAGE_DIAG["diagnosis"]))
    print("\nStage counts with any overlapping candidate at IoU>=0.5:")
    display(STAGE_DIAG.groupby("ROI")[
        ["raw_anyIoU05","author_anyIoU05","nms_anyIoU05"]].sum())
else:
    print("No missed Nadine crowns.")

# Show visually whether a large candidate overlaps multiple Nadine crowns.
# The count is descriptive; Nadine annotations are sparse.
MULTI_ROWS = []
for roi_id in CHECK_ROIS:
    refs = NADINE_MATCHES[(roi_id, "nms_060")]["ref"]
    for stage in ("raw", "author", "nms"):
        if (roi_id, stage) not in SETS:
            continue
        polygons = SETS[(roi_id, stage)]
        for j, poly in enumerate(polygons.geometry):
            n_intersect = sum(
                crown_iou(ref, poly) > 0.05
                for ref in refs.geometry
                if ref.intersects(poly)
            )
            if n_intersect >= 2:
                MULTI_ROWS.append({
                    "ROI": roi_id, "stage": stage, "candidate": j,
                    "Nadine_crowns_intersected": n_intersect,
                    "polygon_m2": round(poly.area, 1)
                })
print("\n=== Saved polygons overlapping two or more Nadine crowns (IoU >0.05) ===")
multi = pd.DataFrame(MULTI_ROWS)
if len(multi):
    display(multi.groupby(["ROI","stage"]).agg(
        polygons=("candidate","count"),
        max_Nadine_crowns=("Nadine_crowns_intersected","max")
    ).reset_index())
else:
    print("No multi-crown polygons detected.")

# Three-stage close-ups for a few informative missed crowns per ROI.
def show_stage_comparison(roi_id, nadine_index):
    ref = NADINE_MATCHES[(roi_id, "nms_060")]["ref"].geometry.iloc[nadine_index]
    x0,y0,x1,y1 = ref.bounds
    cx,cy=(x0+x1)/2,(y0+y1)/2
    half=max((x1-x0)/2,(y1-y0)/2)+8
    bounds=(cx-half,cy-half,cx+half,cy+half)

    with rasterio.open(SITES["Baego"]["rgb"]) as src:
        window=from_bounds(*bounds, transform=src.transform)
        window=window.intersection(Window(0,0,src.width,src.height))
        rgb=src.read([1,2,3],window=window,out_shape=(3,512,512))
        bb=rasterio.windows.bounds(window,src.transform)
    img=np.moveaxis(rgb,0,-1).astype(float)
    lo,hi=np.nanpercentile(img,[2,98])
    img=np.clip((img-lo)/(hi-lo+1e-9),0,1)

    fig,axes=plt.subplots(1,3,figsize=(14,5),constrained_layout=True)
    for ax,stage in zip(axes,("raw","author","nms")):
        ax.imshow(img,extent=[bb[0],bb[2],bb[1],bb[3]])
        polygons=SETS.get((roi_id,stage))
        if polygons is not None:
            nearby=polygons.loc[polygons.geometry.intersects(box(*bb))]
            if len(nearby): nearby.boundary.plot(ax=ax,color="red",linewidth=.8)
            hits=DETAILS[(roi_id,nadine_index)][stage]
            if hits:
                best=polygons.geometry.iloc[hits[0]["index"]]
                gpd.GeoSeries([best],crs=polygons.crs).boundary.plot(
                    ax=ax,color="cyan",linewidth=2)
        gpd.GeoSeries([ref],crs=SITES["Baego"]["crs"]).boundary.plot(
            ax=ax,color="white",linewidth=2)
        best_score=DETAILS[(roi_id,nadine_index)][stage]
        ax.set_title(f"{stage} | best IoU: {best_score[0]['iou']:.3f}" if best_score else f"{stage} | no overlap")
        ax.set_xlim(bb[0],bb[2]);ax.set_ylim(bb[1],bb[3]);ax.set_xticks([]);ax.set_yticks([])
    fig.suptitle(f"Baego ROI {roi_id} | missed Nadine #{nadine_index} | white reference / cyan best / red other")
    plt.show()

for roi_id in CHECK_ROIS:
    if len(STAGE_DIAG)==0: continue
    chosen=STAGE_DIAG.loc[STAGE_DIAG.ROI==roi_id].copy()
    chosen["recoverable"] = chosen["raw_maxIoU"] - chosen["nms_maxIoU"]
    # Largest raw-to-NMS loss first, then highest raw potential
    chosen=chosen.sort_values(["recoverable","raw_maxIoU"],ascending=False)
    for idx in chosen["Nadine_index"].head(INSPECT_PER_ROI):
        show_stage_comparison(roi_id,int(idx))

print("\nInterpretation: compare raw/author/NMS best IoUs; no new segmentation was performed.")


In [ ]:
# 11 — Inspect the original tile-level crown candidates (CPU only)
# Requires Blocks 01, 02, 04, 08. No SAM inference; no files written.
# Archive 'core_*.geojson' might already be filtered: do NOT assume all original SAM masks survive.
from shapely.geometry import box
from pathlib import Path
import geopandas as gpd
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio
from rasterio.windows import Window, from_bounds

STUDY_ROIS = (1, 4)
STAGES = ("author", "nms")
MAX_CASES_PER_ROI = 4     # Limit RGB closeups
TARGET_IOU = 0.5
NEAR_DUPLICATE_IOU = 0.90

def read_tile_candidates(roi_id):
    tile_dir = HISTORY / f"roi{roi_id}_tiles"
    files = sorted(tile_dir.glob("core_*.geojson"))
    if not files:
        raise FileNotFoundError(f"No tile GeoJSONs: {tile_dir}")
    frames, counts = [], []
    raster_crs = SITES["Baego"]["crs"]
    bounds = SITES["Baego"]["bounds"]
    raster_footprint = box(*tuple(bounds))
    for file in files:
        # GeoJSON often silently defaults to EPSG:4326 even when source uses
        # projected XY coordinates. Confirm raw coordinate overlap before overriding.
        g = gpd.read_file(file)
        if g.empty:
            counts.append((file.name, 0, 0))
            continue
        original_bounds = box(*g.total_bounds)
        if not original_bounds.intersects(raster_footprint):
            raise ValueError(
                f"{file.name} doesn't overlap Baego raster in raw XY coordinates. "
                f"Coordinates={tuple(g.total_bounds)}. Check CRS before use."
            )
        g = g.set_crs(raster_crs, allow_override=True)
        g = repaired_copy(g)
        g = g[g.geometry.notna() & ~g.geometry.is_empty].copy()
        g["tile_file"] = file.name
        g["tile_record"] = np.arange(len(g))
        frames.append(g)
        counts.append((file.name, len(g), int((~g.geometry.is_valid).sum())))
    if not frames:
        raise ValueError(f"All tile files empty in {tile_dir}")
    combined = gpd.GeoDataFrame(pd.concat(frames, ignore_index=True),
                                crs=raster_crs)
    return combined, pd.DataFrame(counts,
        columns=["tile_file","n_candidates","n_invalid_after_repair"])

TILE = {}
counts_out=[]
for roi in STUDY_ROIS:
    g, c = read_tile_candidates(roi)
    TILE[roi] = g
    counts_out.append({"ROI":roi, "core_files":len(c),
                       "tile_candidates":len(g),
                       "author_polygons":len(SETS[(roi,"author")]),
                       "nms_polygons":len(SETS[(roi,"nms")])})
    print(f"ROI {roi}: {len(c)} tile files, {len(g)} tile polygons; "
          f"attributes: {list(g.columns)}")
print("\n=== Candidate inventory — tile archive vs merged polygons ===")
display(pd.DataFrame(counts_out))

def best_candidate(ref_geom, g):
    if g is None or len(g)==0 or ref_geom.is_empty:
        return None, 0.
    indices = g.sindex.query(ref_geom, predicate="intersects")
    if not len(indices):
        return None, 0.
    scored = [(int(j), crown_iou(ref_geom, g.geometry.iloc[int(j)]))
              for j in indices]
    return max(scored, key=lambda pair:pair[1])

def max_similar_iou(geom, polygons):
    if not len(polygons): return 0.
    ids = polygons.sindex.query(geom, predicate="intersects")
    return max((crown_iou(geom, polygons.geometry.iloc[int(j)])
                for j in ids), default=0.)

# Audit whether tile candidates were kept in 'author_style',
# and whether a better candidate exists around previously missed crowns.
audit=[]
for roi in STUDY_ROIS:
    tiles=TILE[roi]
    author=SETS[(roi,"author")]
    nms=SETS[(roi,"nms")]
    previous=NADINE_MATCHES[(roi,"nms_060")]
    refs=previous["ref"]
    # Preserve the same missed crown IDs used in Block 09.
    for idx in previous["missed_ref"]:
        rg=refs.geometry.iloc[idx]
        candidates={}
        for label,g in (("tile",tiles),("author",author),("nms",nms)):
            j,iou=best_candidate(rg,g)
            candidates[label]=(j,iou)
        tj,tiou=candidates["tile"]
        if tj is not None:
            besttile=tiles.geometry.iloc[tj]
            similarity=max_similar_iou(besttile,author)
            score=tiles.iloc[tj].get("score",np.nan)
        else:
            similarity=0.;score=np.nan
        if tiou>=TARGET_IOU and candidates["author"][1]<TARGET_IOU:
            diagnosis="Qualified tile mask absent from author output"
        elif tiou>=TARGET_IOU and candidates["nms"][1]<TARGET_IOU:
            diagnosis="Qualified tile mask absent from final NMS"
        elif tiou>candidates["nms"][1]+0.05:
            diagnosis="Better subthreshold tile candidate"
        elif tiou==0:
            diagnosis="No overlapping tile candidate"
        else:
            diagnosis="No improved tile candidate"
        audit.append({
            "ROI":roi, "Nadine_index":idx,
            "tile_best_IoU":round(tiou,3),
            "author_best_IoU":round(candidates["author"][1],3),
            "nms_best_IoU":round(candidates["nms"][1],3),
            "gain_vs_NMS":round(tiou-candidates["nms"][1],3),
            "tile_score":round(float(score),3) if pd.notna(score) else np.nan,
            "tile_best_in_author_IoU":round(similarity,3),
            "tile_best_near_duplicate_in_author":similarity>=NEAR_DUPLICATE_IOU,
            "diagnosis":diagnosis,
            "best_tile_index":tj
        })

TILE_AUDIT=pd.DataFrame(audit)
print("\n=== Previously missed Nadine crowns: tile-stage versus final ===")
display(TILE_AUDIT.sort_values(["ROI","gain_vs_NMS"],
                                ascending=[True,False]).reset_index(drop=True))
print("\n=== Diagnosis counts ===")
display(pd.crosstab(TILE_AUDIT["ROI"],TILE_AUDIT["diagnosis"]))

# Quick global retention summary, using geometry rather than record count alone.
retention=[]
for roi in STUDY_ROIS:
    g=TILE[roi]; author=SETS[(roi,"author")]
    similarities=[max_similar_iou(poly,author) for poly in g.geometry]
    retention.append({
        "ROI":roi, "tile_candidate_count":len(g),
        "tile_equivalent_in_author_IoU90":int(np.sum(np.array(similarities)>=NEAR_DUPLICATE_IOU)),
        "tile_not_equivalent_IoU90":int(np.sum(np.array(similarities)<NEAR_DUPLICATE_IOU)),
        "caution":"This is geometric equivalence, not proof of filtering order"
    })
print("\n=== Geometric retention audit ===")
display(pd.DataFrame(retention))

# Plot the strongest potential recoveries, or largest apparent differences.
def plot_one_case(roi, index):
    ref=NADINE_MATCHES[(roi,"nms_060")]["ref"].geometry.iloc[int(index)]
    x0,y0,x1,y1=ref.bounds
    cx,cy=(x0+x1)/2,(y0+y1)/2
    half=max((x1-x0)/2,(y1-y0)/2)+8
    bounds=(cx-half,cy-half,cx+half,cy+half)
    with rasterio.open(SITES["Baego"]["rgb"]) as ds:
        win=from_bounds(*bounds,transform=ds.transform)
        win=win.intersection(Window(0,0,ds.width,ds.height))
        rgb=ds.read([1,2,3],window=win,out_shape=(3,480,480))
        bb=rasterio.windows.bounds(win,ds.transform)
    im=np.moveaxis(rgb,0,-1).astype(float)
    lo,hi=np.nanpercentile(im,[2,98])
    im=np.clip((im-lo)/(hi-lo+1e-9),0,1)
    fig,axs=plt.subplots(1,3,figsize=(14,5),constrained_layout=True)
    for ax,(label,g) in zip(axs,[
        ("Tile GeoJSON",TILE[roi]),
        ("Author-style",SETS[(roi,"author")]),
        ("NMS",SETS[(roi,"nms")])
    ]):
        ax.imshow(im,extent=(bb[0],bb[2],bb[1],bb[3]))
        nearby=g.loc[g.geometry.intersects(box(*bb))]
        if len(nearby): nearby.boundary.plot(ax=ax,color="red",linewidth=0.75)
        j,iou=best_candidate(ref,g)
        if j is not None:
            gpd.GeoSeries([g.geometry.iloc[j]],crs=g.crs).boundary.plot(
                ax=ax,color="cyan",linewidth=2)
        gpd.GeoSeries([ref],crs=g.crs).boundary.plot(
            ax=ax,color="white",linewidth=2)
        ax.set_title(f"{label} — max IoU {iou:.3f}")
        ax.set_xlim(bb[0],bb[2]);ax.set_ylim(bb[1],bb[3])
        ax.set_xticks([]);ax.set_yticks([])
    fig.suptitle(f"ROI {roi} | missed Nadine #{index}: white=reference, cyan=best candidate, red=others")
    plt.show()

for roi in STUDY_ROIS:
    subset=TILE_AUDIT.loc[TILE_AUDIT.ROI==roi].sort_values(
        ["gain_vs_NMS","tile_best_IoU"],ascending=False)
    for idx in subset["Nadine_index"].head(MAX_CASES_PER_ROI):
        plot_one_case(roi,int(idx))

print("\nBlock 11 complete. Tile GeoJSONs are archived candidates; their original SAM filtering status is unverified.")
